# ViT + D2NN CIFAR10 对比（v3：贴近原始 single-layer 结构）

本版本目标：
- **尽量保留** `D2NN-single-layer-CIFAR10(FO).ipynb` 的原始训练结构和思路。
- 只做必要改动：
  1) D2NN层数改为12层；
  2) detector输出改为共享FC头输出；
  3) 增加ViT-Base + 同一个SharedFCHead用于对比（可选运行）。

In [ ]:
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16
from tqdm.auto import tqdm

plt.rcParams['font.sans-serif'] = ['SimHei', 'Noto Sans CJK SC', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('device =', device)

In [ ]:
# ========== 配置（基本沿用原single-layer写法） ==========
BATCH_SIZE = 200
IMG_SIZE = 64
N_pixels = 128
PADDING = (N_pixels - IMG_SIZE) // 2

# D2NN核心参数（与原始风格一致）
wl = 700e-9
pixel_size = 2e-6
num_layers = 12

# 两阶段训练参数（保持原始思想：先phase，再全部解禁）
stage1_epochs = 8
stage2_epochs = 12
stage1_lr = 1e-4
stage2_lr = 1e-6

# ViT开关（你主要看D2NN，可设False不跑）
RUN_VIT = False

In [ ]:
# ========== 数据处理（沿用原single-layer风格） ==========
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Pad([PADDING, PADDING], fill=(0), padding_mode='constant'),
])

train_dataset = torchvision.datasets.CIFAR10('./data', train=True, transform=transform, download=True)
val_dataset = torchvision.datasets.CIFAR10('./data', train=False, transform=transform, download=True)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = train_dataset.classes
print('train size:', len(train_dataset), 'val size:', len(val_dataset))

In [ ]:
# ========== 共用FC头（只新增，不破坏原D2NN主体） ==========
class SharedFCHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, num_classes=10):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.head(x)

In [ ]:
# ========== ViT + 共用头（额外模块，可选运行） ==========
class ViTWithSharedHead(nn.Module):
    def __init__(self, num_classes=10, fc_hidden_dim=512):
        super().__init__()
        self.backbone = vit_b_16(weights=None)
        in_dim = self.backbone.heads.head.in_features
        self.backbone.heads = nn.Identity()
        self.fc_head = SharedFCHead(in_dim, hidden_dim=fc_hidden_dim, num_classes=num_classes)

    def forward(self, x):
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        feat = self.backbone(x)
        return self.fc_head(feat)

In [ ]:
# ========== D2NN原有基础模块（尽量照抄single-layer风格） ==========
class Diffractive_Layer(torch.nn.Module):
    def __init__(self, λ=torch.tensor([700, 546.1, 435.8])*1e-9, N_pixels=128, pixel_size=2e-6, distance=torch.tensor([0.005])):
        super(Diffractive_Layer, self).__init__()

        fx = torch.fft.fftshift(torch.fft.fftfreq(N_pixels, d=pixel_size))
        fy = torch.fft.fftshift(torch.fft.fftfreq(N_pixels, d=pixel_size))
        fxx, fyy = torch.meshgrid(fx, fy, indexing='ij')

        λ = λ.unsqueeze(1).repeat(1, 1).unsqueeze(1).repeat(1, 1, 1)
        fxx = fxx.unsqueeze(0).expand([len(λ), N_pixels, -1])
        fyy = fyy.unsqueeze(0).expand([len(λ), N_pixels, -1])

        argument = (2 * np.pi)**2 * ((1. / λ) ** 2 - fxx ** 2 - fyy ** 2)

        tmp = np.sqrt(np.abs(argument))
        self.distance = distance.to(device)
        self.kz = torch.tensor(np.where(argument >= 0, tmp, 1j * tmp)).to(device)

    def forward(self, E):
        fft_c = torch.fft.fft2(E)
        c = torch.fft.fftshift(fft_c, dim=(1, 2))
        phase = torch.exp(1j * self.kz * self.distance).to(device)
        angular_spectrum = torch.fft.ifft2(torch.fft.ifftshift(c * phase, dim=(1, 2)))
        return angular_spectrum

In [ ]:
# ========== DNN主体（保持原结构，只把detector替换为共享FC） ==========
class DNN(torch.nn.Module):
    def __init__(self, phase=[], num_layers=12, wl=700e-9, N_pixels=128, pixel_size=2e-6, distance=[]):
        super(DNN, self).__init__()

        self.num_layers = num_layers
        self.diffractive_layers = torch.nn.ModuleList([
            Diffractive_Layer(torch.tensor([700, 546.1, 435.8])*1e-9, N_pixels, pixel_size, distance[i])
            for i in range(num_layers)
        ])
        self.last_diffractive_layer = Diffractive_Layer(torch.tensor([700, 546.1, 435.8])*1e-9, N_pixels, pixel_size, distance[-1])

        for i in range(num_layers):
            self.register_parameter('phase_' + str(i), phase[i])

        # 只替换输出头：detector -> shared fc
        self.pool = nn.AdaptiveAvgPool2d((8, 8))
        self.fc_head = SharedFCHead(3*8*8, hidden_dim=512, num_classes=10)

    def forward(self, E):
        # 与原代码一致：输入先转振幅
        E = torch.sqrt(torch.squeeze(E)).to(torch.complex64)

        for index, layer in enumerate(self.diffractive_layers):
            temp = layer(E)
            phase = getattr(self, 'phase_' + str(index))
            constr_phase = 2 * torch.pi * torch.sigmoid(phase)
            exp_j_phase = torch.exp(1j * constr_phase)
            E = temp * exp_j_phase

        E = self.last_diffractive_layer(E)
        Int = torch.abs(E) ** 2

        feat = self.pool(Int).flatten(1)
        out = self.fc_head(feat)
        return out, Int

In [ ]:
# ========== 初始化phase和distance（风格贴近原始） ==========
phase = [torch.nn.Parameter(torch.from_numpy(np.random.random(size=(N_pixels, N_pixels)).astype(np.float32)).to(device))
         for _ in range(num_layers)]

distance = [torch.nn.Parameter(torch.tensor([0.005], dtype=torch.float32, device=device)) for _ in range(num_layers + 1)]

model = DNN(phase=phase, num_layers=num_layers, wl=wl, N_pixels=N_pixels, pixel_size=pixel_size, distance=distance).to(device)

In [ ]:
# ========== 训练与评估（保持原有习惯，evaluate返回tuple） ==========
def evaluate(model, loader, criterion):
    model.eval()
    total = 0
    correct = 0
    ep_loss = 0.0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device)
            labels = labels.to(device)

            out_labels, _ = model(images)
            probs = torch.softmax(out_labels, dim=1)
            labels_onehot = F.one_hot(labels, num_classes=10).float()

            loss = criterion(probs, labels_onehot)
            ep_loss += loss.item()

            _, predicted = torch.max(probs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            preds_all.append(predicted.detach().cpu())
            labels_all.append(labels.detach().cpu())

    acc = correct / total
    avg_loss = ep_loss / max(len(loader), 1)
    preds_all = torch.cat(preds_all) if preds_all else torch.tensor([])
    labels_all = torch.cat(labels_all) if labels_all else torch.tensor([])
    return acc, avg_loss, preds_all, labels_all


def train(model, criterion, optimizer, trainloader, testloader, epochs=10):
    train_loss_hist, train_acc_hist = [], []
    test_loss_hist, test_acc_hist = [], []
    best_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        ep_loss = 0.0
        train_total = 0
        train_correct = 0

        for images, labels in tqdm(trainloader, desc=f'Epoch {epoch+1}/{epochs}'):
            images = images.to(device)
            labels = labels.to(device)
            labels_onehot = F.one_hot(labels, num_classes=10).float()

            out_labels, _ = model(images)
            probs = torch.softmax(out_labels, dim=1)
            loss = criterion(probs, labels_onehot)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            ep_loss += loss.item()
            _, predicted = torch.max(probs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        tr_loss = ep_loss / max(len(trainloader), 1)
        tr_acc = train_correct / max(train_total, 1)

        eval_out = evaluate(model, testloader, criterion)
        te_acc = eval_out[0]   # 注意：evaluate返回tuple，第一位是acc
        te_loss = eval_out[1]

        train_loss_hist.append(tr_loss)
        train_acc_hist.append(tr_acc)
        test_loss_hist.append(te_loss)
        test_acc_hist.append(te_acc)

        if te_acc >= best_acc:
            best_acc = te_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'Epoch={epoch} train loss={tr_loss:.4f}, test loss={te_loss:.4f}')
        print(f'train acc={tr_acc:.4f}, test acc={te_acc:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_loss_hist, train_acc_hist, test_loss_hist, test_acc_hist, model

In [ ]:
# ========== Stage-1：先只训练相位 + FC头（保留核心思想） ==========
for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if ('phase_' in name) or ('fc_head' in name):
        param.requires_grad = True

print('Params to learn in stage1:')
params_to_update = []
for name, param in model.named_parameters():
    if param.requires_grad:
        params_to_update.append(param)
        print('	', name)

criterion = torch.nn.MSELoss(reduction='sum').to(device)
optimizer = torch.optim.Adam(params_to_update, lr=stage1_lr)

train_loss_s1, train_acc_s1, test_loss_s1, test_acc_s1, model = train(
    model, criterion, optimizer, train_dataloader, val_dataloader, epochs=stage1_epochs
)

In [ ]:
# ========== Stage-2：全部解禁，相位和层间距都拿来训练（沿用原思路） ==========
for param in model.parameters():
    param.requires_grad = True

print('Params to learn in stage2:')
params_to_update = []
for name, param in model.named_parameters():
    if param.requires_grad:
        params_to_update.append(param)
        print('	', name)

criterion = torch.nn.MSELoss(reduction='sum').to(device)
optimizer = torch.optim.Adam(params_to_update, lr=stage2_lr)

train_loss_s2, train_acc_s2, test_loss_s2, test_acc_s2, model = train(
    model, criterion, optimizer, train_dataloader, val_dataloader, epochs=stage2_epochs
)

In [ ]:
# ========== 合并并绘图（保证使用最新数据） ==========
all_train_loss = train_loss_s1 + train_loss_s2
all_test_loss = test_loss_s1 + test_loss_s2
all_train_acc = train_acc_s1 + train_acc_s2
all_test_acc = test_acc_s1 + test_acc_s2

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(all_train_loss, marker='o', label='train_loss')
plt.plot(all_test_loss, marker='o', label='test_loss')
plt.axvline(len(train_loss_s1)-1, color='r', linestyle='--', label='stage1->stage2')
plt.title('D2NN损失曲线（最新轮次）')
plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(np.array(all_train_acc)*100, marker='o', label='train_acc')
plt.plot(np.array(all_test_acc)*100, marker='o', label='test_acc')
plt.axvline(len(train_acc_s1)-1, color='r', linestyle='--', label='stage1->stage2')
plt.title('D2NN准确率曲线（最新轮次）')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'✅ D2NN最终best test acc(估计): {max(all_test_acc)*100:.2f}%')

In [ ]:
# ========== 混淆矩阵（使用最终模型最新输出） ==========
def confusion_matrix(predicted, labels, conf_matrix):
    for p, t in zip(predicted, labels):
        conf_matrix[t, p] += 1
    return conf_matrix

conf_matrix = torch.zeros(10, 10)

model.eval()
with torch.no_grad():
    for images, labels in tqdm(val_dataloader, desc='CM'):
        images = images.to(device)
        labels = labels.to(device)
        out_labels, _ = model(images)
        probs = torch.softmax(out_labels, dim=1)
        _, predicted = torch.max(probs.data, 1)
        conf_matrix = confusion_matrix(predicted.cpu(), labels.cpu(), conf_matrix)

cm = np.array(conf_matrix)
plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap='Blues')
plt.title('D2NN(12层+共享FC头) 混淆矩阵')
plt.colorbar()
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.tight_layout()
plt.show()

In [ ]:
# ========== 可选：ViT训练（默认不跑） ==========
def train_simple_cls(model, trainloader, testloader, epochs=5, lr=1e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    best_acc = 0.0

    for ep in range(epochs):
        model.train()
        for images, labels in tqdm(trainloader, desc=f'ViT {ep+1}/{epochs}'):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        total, correct = 0, 0
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                pred = model(images).argmax(1)
                total += labels.size(0)
                correct += (pred == labels).sum().item()
        acc = correct / total
        best_acc = max(best_acc, acc)
        print(f'ViT epoch={ep+1} test_acc={acc*100:.2f}%')

    print(f'✅ ViT best_acc={best_acc*100:.2f}%')

if RUN_VIT:
    vit_model = ViTWithSharedHead(num_classes=10, fc_hidden_dim=512)
    train_simple_cls(vit_model, train_dataloader, val_dataloader, epochs=5, lr=1e-4)

In [ ]:
# ========== 保存结果 ==========
result = {
    'stage1_best_test_acc': float(max(test_acc_s1) if len(test_acc_s1) else 0.0),
    'stage2_best_test_acc': float(max(test_acc_s2) if len(test_acc_s2) else 0.0),
    'all_best_test_acc': float(max(all_test_acc) if len(all_test_acc) else 0.0),
    'num_layers': num_layers,
    'fc_head_shared': True,
}

os.makedirs('results', exist_ok=True)
with open('results/cifar10_d2nn12_sharedfc_v3.json', 'w', encoding='utf-8') as f:
    import json
    json.dump(result, f, ensure_ascii=False, indent=2)

print('✅ 保存完成:', result)